# Market efficiency and a state-space model of team strength
### Serie A, 2018/19 - 2026/27

Companion notebook to the paper. Run the cells in order; each cell prints the tables and saves the figures used in the text (PNG and PDF, white background).

Data: football-data.co.uk, one CSV per season, all in the folder set in `DATA_DIR`.

# CELL 1 - Setup and data
Serie A match data from football-data.co.uk. One CSV per season.
The season is taken from the FIRST date of each file, not from each match date,
because the 2019/20 season ran until August 2020 (Covid).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq, minimize, minimize_scalar
from scipy.stats import poisson, skellam, norm, chi2, t as student_t

DATA_DIR = os.path.join('..', 'data')   # folder with the CSV files
FIG_DIR = 'figures'                                     # figures for the paper are saved here
os.makedirs(FIG_DIR, exist_ok=True)

# plain white style for the paper
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'font.size': 9,
                     'axes.grid': True, 'grid.alpha': 0.3, 'grid.linewidth': 0.5,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'savefig.bbox': 'tight'})
BLUE, ORANGE, GREY, RED = '#1f77b4', '#ff7f0e', '#7f7f7f', '#d62728'


def save_fig(fig, name):
    """Save a figure in both formats: PNG to look at, PDF for LaTeX."""
    fig.savefig(f'{FIG_DIR}/{name}.png')
    fig.savefig(f'{FIG_DIR}/{name}.pdf')


COLS = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HST', 'AST',
        'PSH', 'PSD', 'PSA', 'PSCH', 'PSCD', 'PSCA', 'PC>2.5', 'PC<2.5',
        'BFEH', 'BFED', 'BFEA', 'BFECH', 'BFECD', 'BFECA', 'BFEC>2.5', 'BFEC<2.5',
        'B365H', 'B365D', 'B365A', 'HxG', 'AxG']

frames = []
for f in sorted(os.listdir(DATA_DIR)):
    if not f.endswith('.csv'):
        continue
    d = pd.read_csv(os.path.join(DATA_DIR, f), encoding='utf-8-sig')
    d = d.reindex(columns=COLS).dropna(subset=['HomeTeam', 'AwayTeam', 'FTHG'])
    d['Date'] = pd.to_datetime(d['Date'], dayfirst=True, format='mixed')
    start = d['Date'].min()
    d['season'] = start.year if start.month >= 7 else start.year - 1
    frames.append(d)

df = pd.concat(frames, ignore_index=True).sort_values('Date', kind='stable').reset_index(drop=True)
assert df.groupby('season').size().max() <= 380, 'two files map to the same season'

summary = df.groupby('season').agg(
    matches=('HomeTeam', 'size'), teams=('HomeTeam', 'nunique'),
    goals_per_team=('FTHG', lambda x: (x.mean() + df.loc[x.index, 'FTAG'].mean()) / 2),
    shots_on_target=('HST', lambda x: (x.mean() + df.loc[x.index, 'AST'].mean()) / 2),
    home_win_rate=('FTR', lambda x: (x == 'H').mean()))
summary['pinnacle_1x2'] = df.groupby('season')['PSCH'].apply(lambda x: x.notna().mean())
print('TABLE 1 - Data by season')
print(summary.round(3).to_string())

# CELL 2 - The market as a probability measure
Decimal odds imply probabilities that sum to more than one. The excess is the
bookmaker margin, removed here in proportion. We then check whether the resulting
probabilities are well calibrated: a 30% event should happen about 30% of the time.

In [ ]:
SEASONS_MKT = [2020, 2021, 2022, 2023, 2024]          # seasons with complete Pinnacle closing odds
m = df[df['season'].isin(SEASONS_MKT) & df['PSCH'].notna() & df['PC>2.5'].notna()].copy()


def devig(odds):
    """Proportional de-vigging: probabilities are scaled so that they sum to one."""
    inv = 1 / np.asarray(odds, dtype=float)
    return inv / inv.sum(axis=1, keepdims=True)


p_1x2 = devig(m[['PSCH', 'PSCD', 'PSCA']])
m['p1'], m['pX'], m['p2'] = p_1x2.T
m['pOver'] = devig(m[['PC>2.5', 'PC<2.5']])[:, 0]
margin = (1 / m[['PSCH', 'PSCD', 'PSCA']]).sum(axis=1) - 1
print(f'Pinnacle closing margin: mean {margin.mean():.3f}, range {margin.min():.3f}-{margin.max():.3f}, '
      f'n = {len(m)}')

# calibration: group matches by predicted probability and compare with the observed frequency
pred = np.r_[m['p1'], m['p2']]
obs = np.r_[(m['FTR'] == 'H').astype(int), (m['FTR'] == 'A').astype(int)]
bins = pd.qcut(pred, 10, labels=False)
cal = pd.DataFrame({'pred': pred, 'obs': obs, 'bin': bins}).groupby('bin').agg(
    n=('obs', 'size'), predicted=('pred', 'mean'), observed=('obs', 'mean'))
cal['se'] = np.sqrt(cal['observed'] * (1 - cal['observed']) / cal['n'])

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].plot([0, 1], [0, 1], color=GREY, ls='--', lw=1, label='perfect calibration')
ax[0].errorbar(cal['predicted'], cal['observed'], yerr=1.96 * cal['se'], fmt='o',
               color=BLUE, ms=5, capsize=3, label='Pinnacle closing')
ax[0].set(xlabel='predicted probability', ylabel='observed frequency', xlim=(0, 0.95), ylim=(0, 0.95))
ax[0].legend(frameon=False)
ax[0].set_title('(a) Calibration of 1X2 probabilities')

x = m['p1'] - m['p2']
y = m['FTHG'] - m['FTAG']
slope, intercept = np.polyfit(x, y, 1)
r2 = np.corrcoef(x, y)[0, 1] ** 2
ax[1].scatter(x, y, s=4, alpha=0.15, color=BLUE)
xx = np.linspace(x.min(), x.max(), 2)
ax[1].plot(xx, intercept + slope * xx, color=RED, lw=2,
           label=f'slope {slope:.2f}, $R^2$ {r2:.2f}')
ax[1].set(xlabel='P(home) - P(away)', ylabel='goal difference')
ax[1].legend(frameon=False)
ax[1].set_title('(b) Goal difference against market')
fig.tight_layout()
save_fig(fig, 'fig1_calibration')
plt.show()

# CELL 3 - Implied goal intensities
Same idea as implied volatility: we invert a pricing model until it reproduces the
market prices. Two prices (Over 2.5 and the 1X2 spread) give two intensities.
The draw is NOT used, so it works as an out-of-sample check on the model.

In [ ]:
def total_from_over(p_over):
    """Total goals T such that P(total >= 3) equals the market Over probability."""
    return brentq(lambda T: (1 - poisson.cdf(2, T)) - p_over, 0.01, 10)


def split_from_1x2(T, p1, p2):
    """Share s of T to the home team such that P(home) - P(away) matches the market."""
    def err(s):
        lh, la = s * T, (1 - s) * T
        return (skellam.sf(0, lh, la) - skellam.cdf(-1, lh, la)) - (p1 - p2)
    if err(0.001) * err(0.999) > 0:
        return np.nan
    return brentq(err, 0.001, 0.999)


def implied_intensities(p1, p2, p_over):
    """Two market prices give the two goal intensities, uniquely."""
    T = np.array([total_from_over(p) for p in p_over])
    s = np.array([split_from_1x2(t, a, b) for t, a, b in zip(T, p1, p2)])
    return s * T, (1 - s) * T, T


m['lam_h'], m['lam_a'], T = implied_intensities(m['p1'], m['p2'], m['pOver'])

# independent Poisson: what does it say about the draw and about both teams scoring?
m['pX_poisson'] = skellam.pmf(0, m['lam_h'], m['lam_a'])
m['pBTTS_poisson'] = (1 - np.exp(-m['lam_h'])) * (1 - np.exp(-m['lam_a']))

# Dixon-Coles: the implied rho that reproduces the market draw, in closed form
m['rho_implied'] = (m['pX_poisson'] - m['pX']) / (2 * m['lam_h'] * m['lam_a'] * np.exp(-T))
RHO = m.loc[m['season'].isin([2021, 2022]), 'rho_implied'].median()      # fixed on two seasons only
m['pBTTS_dc'] = m['pBTTS_poisson'] - RHO * m['lam_h'] * m['lam_a'] * np.exp(-T)


def bivariate_lambdas(p1, p2, p_over):
    """Bivariate Poisson: goals are X1+X3 and X2+X3. The common part l3 raises the total
    without changing the goal difference, so 1X2 and Over can be matched exactly."""
    def split(S):
        return brentq(lambda z: (skellam.sf(0, z * S, (1 - z) * S)
                                 - skellam.cdf(-1, z * S, (1 - z) * S)) - (p1 - p2), 1e-4, 1 - 1e-4)

    def draw_err(S):
        z = split(S)
        return skellam.pmf(0, z * S, (1 - z) * S) - (1 - p1 - p2)

    S_min = -np.log(1 - abs(p1 - p2)) + 0.05          # below this the 1X2 cannot be reproduced
    if draw_err(S_min) * draw_err(8) > 0:
        return np.nan, np.nan, np.nan
    S = brentq(draw_err, S_min, 8)
    l1, l2 = split(S) * S, (1 - split(S)) * S

    def over_err(l3):
        return 1 - np.exp(-l3) * (poisson.cdf(2, S) + l3 * np.exp(-S)) - p_over

    if over_err(0) > 0:                                # already too much Over with l3 = 0
        return l1, l2, np.nan
    return l1, l2, brentq(over_err, 0, 5)


biv = np.array([bivariate_lambdas(a, b, o) for a, b, o in zip(m['p1'], m['p2'], m['pOver'])])
m['l1'], m['l2'], m['l3'] = biv.T
m['pBTTS_biv'] = (1 - np.exp(-(m['l1'] + m['l3'])) - np.exp(-(m['l2'] + m['l3']))
                  + np.exp(-(m['l1'] + m['l2'] + m['l3'])))

btts = ((m['FTHG'] > 0) & (m['FTAG'] > 0)).astype(float)
tab2 = pd.DataFrame({
    'draw: model - market (pp)': [100 * (m['pX_poisson'] - m['pX']).mean(), 0.0, 0.0],
    'BTTS: model - observed (pp)': [100 * (m['pBTTS_poisson'] - btts).mean(),
                                    100 * (m['pBTTS_dc'] - btts).mean(),
                                    100 * (m['pBTTS_biv'] - btts).mean()],
}, index=['independent Poisson', 'Dixon-Coles', 'bivariate Poisson'])
print(f'\nTABLE 2 - Models against market prices (n = {len(m)}, implied rho = {RHO:.3f}, '
      f'median l3 = {m["l3"].median():.3f}, no solution in {m["l3"].isna().sum()} matches)')
print(tab2.round(2).to_string())

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].hist(m['rho_implied'].dropna(), bins=40, color=BLUE, alpha=0.8)
ax[0].axvline(RHO, color=RED, lw=2, label=f'median {RHO:.3f}')
ax[0].set(xlabel=r'implied $\rho$ (Dixon-Coles)', ylabel='matches', xlim=(-0.3, 0.05))
ax[0].legend(frameon=False)
ax[0].set_title('(a) Dependence implied by the draw price')

for col, lab, c in [('pBTTS_poisson', 'independent Poisson', GREY),
                    ('pBTTS_dc', 'Dixon-Coles', ORANGE), ('pBTTS_biv', 'bivariate', BLUE)]:
    by_season = (m.groupby('season').apply(lambda g: 100 * (g[col] - btts.loc[g.index]).mean(),
                                           include_groups=False))
    ax[1].plot(by_season.index, by_season.values, 'o-', color=c, label=lab)
ax[1].axhline(0, color='black', lw=0.8)
ax[1].set(xlabel='season', ylabel='BTTS: model - observed (pp)')
ax[1].legend(frameon=False)
ax[1].set_title('(b) Both teams to score, by season')
fig.tight_layout()
save_fig(fig, 'fig2_goal_models')
plt.show()

# CELL 4 - Is there momentum the market misses?
For every team we build the gap between what happened and what the market expected.
If a "hot streak" existed and the market ignored it, these gaps would be autocorrelated.
The null distribution comes from shuffling the order of matches WITHIN each team-season:
this keeps the level of each team and destroys only the time ordering.

In [ ]:
SEASONS_SHOTS = [2021, 2022, 2023, 2024]              # 2020/21 excluded: shots counted differently
ps = m[m['season'].isin(SEASONS_SHOTS)].reset_index(drop=True)

long = pd.concat([
    pd.DataFrame({'team': ps['HomeTeam'], 'season': ps['season'], 'date': ps['Date'],
                  'GF': ps['FTHG'], 'GA': ps['FTAG'], 'SF': ps['HST'], 'SA': ps['AST'],
                  'lam_f': ps['lam_h'], 'lam_a': ps['lam_a']}),
    pd.DataFrame({'team': ps['AwayTeam'], 'season': ps['season'], 'date': ps['Date'],
                  'GF': ps['FTAG'], 'GA': ps['FTHG'], 'SF': ps['AST'], 'SA': ps['HST'],
                  'lam_f': ps['lam_a'], 'lam_a': ps['lam_h']})
]).sort_values(['season', 'team', 'date']).reset_index(drop=True)
long['group'] = long['season'].astype(str) + ' ' + long['team']

# expected shots on target from the market intensity, one regression per season
coef = {s: np.polyfit(g['lam_f'], g['SF'], 1) for s, g in long.groupby('season')}
b_s = long['season'].map({s: c[0] for s, c in coef.items()})
a_s = long['season'].map({s: c[1] for s, c in coef.items()})
conv = long.groupby('season').apply(lambda g: g['GF'].sum() / g['SF'].sum(), include_groups=False)
c_s = long['season'].map(conv)

long['goals_for'] = long['GF'] - long['lam_f']
long['goals_against'] = long['GA'] - long['lam_a']
long['shots_for'] = long['SF'] - (a_s + b_s * long['lam_f'])
long['shots_against'] = long['SA'] - (a_s + b_s * long['lam_a'])
long['conversion'] = long['GF'] - c_s * long['SF']
SERIES = ['goals_for', 'goals_against', 'shots_for', 'shots_against', 'conversion']

code = pd.factorize(long['group'])[0]
LAGS, N_PERM = [1, 2, 3, 4, 5], 1000
rng = np.random.default_rng(2026)


def acf_within(r, k):
    """Autocorrelation at lag k, pairs taken only inside the same team-season."""
    same = code[k:] == code[:-k]
    return (r[k:] * r[:-k])[same].sum() / (r ** 2).sum()


def shuffle_within():
    """Indices that reshuffle the order inside each team-season."""
    return np.lexsort((rng.random(len(code)), code))


rows, acf_obs, acf_null = [], {}, {}
for name in SERIES:
    r = long[name].to_numpy()
    obs = np.array([acf_within(r, k) for k in LAGS])
    null = np.array([[acf_within(r[shuffle_within()], k) for k in LAGS] for _ in range(N_PERM)])
    acf_obs[name], acf_null[name] = obs, null
    p = (1 + (null.mean(axis=1) >= obs.mean()).sum()) / (N_PERM + 1)
    rows.append({'series': name, 'mean ACF (lags 1-5)': obs.mean(),
                 'chance level': null.mean(), 'p-value': p})
tab3 = pd.DataFrame(rows).set_index('series')
print(f'\nTABLE 3 - Persistence of gaps against the market ({len(long)} team-matches, '
      f'{long["group"].nunique()} team-seasons, {N_PERM} permutations)')
print(tab3.round(4).to_string())
print(f'threshold with {len(SERIES)} tests (Bonferroni): p < {0.05 / len(SERIES):.4f}')

fig, axes = plt.subplots(1, len(SERIES), figsize=(12, 2.8), sharey=True)
for ax, name in zip(axes, SERIES):
    lo = np.percentile(acf_null[name], 2.5, axis=0)
    hi = np.percentile(acf_null[name], 97.5, axis=0)
    ax.fill_between(LAGS, lo, hi, color=GREY, alpha=0.3, label='chance (95%)')
    ax.bar(LAGS, acf_obs[name], color=BLUE, width=0.6, label='data')
    ax.axhline(0, color='black', lw=0.8)
    ax.set(xlabel='lag (matches)', title=name.replace('_', ' '), xticks=LAGS)
axes[0].set_ylabel('autocorrelation')
axes[0].legend(frameon=False, fontsize=7)
fig.suptitle('Gaps against the market show no persistence', y=1.04)
fig.tight_layout()
save_fig(fig, 'fig3_persistence')
plt.show()

# CELL 5 - A state-space model for team strength

Strength is not observed: we only see noisy matches. The local level model treats strength
as a hidden state that may drift, and the Kalman filter updates it after every match.

**Every number in the model is estimated from the data**, none is chosen by hand: Q (drift
between matches), R (noise of a single match), h (home advantage), Q_summer (drift over the
summer) and P0 (how uncertain the filter is before it has seen anything). They are all
obtained by maximum likelihood through the prediction error decomposition.

In [ ]:
SHOTS_DATA = df[(df['season'] >= 2021) & df['HST'].notna()].copy()
TEAMS = sorted(set(SHOTS_DATA['HomeTeam']) | set(SHOTS_DATA['AwayTeam']))
IDX = {t: i for i, t in enumerate(TEAMS)}
IN_SEASON = {s: set(g['HomeTeam'].map(IDX)) for s, g in SHOTS_DATA.groupby('season')}
PARAM_NAMES = ['Q', 'R', 'h', 'Q_summer', 'P0']     # every number below is estimated, none is chosen by hand


def kalman_league(d, par):
    """Kalman filter with all teams in the state vector.
    Each match observes one contrast: shots_home - shots_away = b_i - b_j + h + noise.
    par: dict with Q (drift per match), R (match noise), h (home advantage),
         Q_summer (drift over the summer), P0 (prior variance on strength)."""
    n, k = len(d), len(TEAMS)
    home, away = d['HomeTeam'].map(IDX).to_numpy(), d['AwayTeam'].map(IDX).to_numpy()
    y = (d['HST'] - d['AST']).to_numpy(float)
    season = d['season'].to_numpy()
    b, P = np.zeros(k), np.eye(k) * par['P0']            # prior: everyone average, same uncertainty
    pred, F, v = np.empty(n), np.empty(n), np.empty(n)
    B, PP = np.empty((n, k)), np.empty((n, k, k))
    for t in range(n):
        if t > 0 and season[t] != season[t - 1]:                      # summer break
            before, now = IN_SEASON[season[t - 1]], IN_SEASON[season[t]]
            for i in before & now:
                P[i, i] += par['Q_summer']                            # teams that stay: more uncertainty
            relegated_mean = np.mean([b[i] for i in before - now])
            for i in now - before:                                    # promoted teams start over
                b[i], P[i, :], P[:, i] = relegated_mean, 0.0, 0.0
                P[i, i] = par['P0']
        i, j = home[t], away[t]
        P[i, i] += par['Q']                                           # predict: only the two teams playing
        P[j, j] += par['Q']
        B[t], PP[t] = b, P
        pred[t] = b[i] - b[j] + par['h']
        F[t] = P[i, i] + P[j, j] - 2 * P[i, j] + par['R']             # variance of the surprise
        v[t] = y[t] - pred[t]
        gain = (P[:, i] - P[:, j]) / F[t]                             # Kalman gain
        b = b + gain * v[t]
        P = P - np.outer(gain, gain) * F[t]
    return dict(y=y, pred=pred, F=F, v=v, B=B, PP=PP, season=season)


def log_likelihood(par, d, mask=None):
    """Prediction error decomposition: each surprise v is N(0, F) if the model is right."""
    k = kalman_league(d, par)
    sel = slice(None) if mask is None else mask
    F, v = k['F'][sel], k['v'][sel]
    return -0.5 * np.sum(np.log(2 * np.pi * F) + v ** 2 / F)


def unpack(x, fixed=None):
    """Variances are estimated in logs so that they stay positive; h is free."""
    p = dict(Q=np.exp(x[0]), R=np.exp(x[1]), h=x[2], Q_summer=np.exp(x[3]), P0=np.exp(x[4]))
    return {**p, **(fixed or {})}


def fit_params(d, mask=None, fixed=None):
    """Maximum likelihood on all five parameters. 'fixed' pins some of them for a restricted model."""
    x0 = [np.log(0.05), np.log(8.0), 0.5, np.log(0.5), np.log(2.0)]
    res = minimize(lambda x: -log_likelihood(unpack(x, fixed), d, mask), x0, method='Nelder-Mead',
                   options=dict(xatol=1e-4, fatol=1e-4, maxiter=6000))
    free = 5 - len(fixed or {})
    return {**unpack(res.x, fixed), 'logL': -res.fun, 'k': free, 'n': len(d) if mask is None else int(mask.sum())}


fit_data = SHOTS_DATA[SHOTS_DATA['season'].isin([2021, 2022])].reset_index(drop=True)
par = fit_params(fit_data)
print(f'TABLE 4 - Parameters estimated on 2021/22-2022/23 ({len(fit_data)} matches), '
      f'all by maximum likelihood')
meaning = {'Q': 'strength drift between two matches', 'R': 'noise of a single match',
           'h': 'home advantage in shots on target', 'Q_summer': 'strength drift over the summer',
           'P0': 'prior uncertainty on strength, before any match'}
print(pd.DataFrame({'estimate': [par[k] for k in PARAM_NAMES],
                    'meaning': [meaning[k] for k in PARAM_NAMES]}, index=PARAM_NAMES).round(4).to_string())
print(f'single match noise: +/- {np.sqrt(par["R"]):.2f} shots on target; '
      f'prior spread of strengths: +/- {np.sqrt(par["P0"]):.2f}')

# --- is the drift Q really zero? Q = 0 sits on the boundary, so the null is a 50:50 mixture
par_Q0 = fit_params(fit_data, fixed={'Q': 1e-10})
LR = 2 * (par['logL'] - par_Q0['logL'])
p_LR = 0.5 * chi2.sf(max(LR, 0), 1)
grid_Q = np.geomspace(1e-4, 0.1, 25)
profile = np.array([log_likelihood({**par, 'Q': q}, fit_data) for q in grid_Q]) - par['logL']
Q_max = grid_Q[profile >= -1.92].max()           # chi2(1) 95% / 2
print(f'\ntest Q = 0: LR = {LR:.2f}, p = {p_LR:.3f}  |  Q compatible with the data (95%): '
      f'0 to {Q_max:.3f}, i.e. at most +/- {np.sqrt(38 * Q_max):.2f} shots of drift over a season')

kf = kalman_league(fit_data, par)
z = kf['v'] / np.sqrt(kf['F'])
print(f'standardised surprises in sample: mean {z.mean():+.3f}, variance {z.var():.3f} (ideal 0 and 1)')


def centred_strength(k, t):
    """Strength as a deviation from the league mean. The data only see differences,
    so the common level is not identified and must be removed from the bands too."""
    league = sorted(IN_SEASON[k['season'][t]])
    C = -np.ones((len(league), len(TEAMS))) / len(league)
    C[np.arange(len(league)), league] += 1
    return league, C @ k['B'][t], np.sqrt(np.diag(C @ k['PP'][t] @ C.T))


fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].plot(grid_Q, profile, color=BLUE, lw=2)
ax[0].axhline(-1.92, color=GREY, ls=':', lw=1.2, label='95% threshold (-1.92)')
ax[0].axvline(Q_max, color=ORANGE, ls='--', lw=1.2, label=f'upper limit {Q_max:.3f}')
ax[0].set(xscale='log', xlabel='Q (strength drift per match)',
          ylabel='log-likelihood (0 = maximum)', ylim=(-12, 0.5))
ax[0].legend(frameon=False, loc='lower left')
ax[0].set_title('(a) How much does strength move?')

HIGHLIGHT = {'Inter': BLUE, 'Napoli': ORANGE, 'Salernitana': RED}
rows_tr = []
for t in range(len(fit_data)):
    league, f_, sd_ = centred_strength(kf, t)
    for team in (fit_data.at[t, 'HomeTeam'], fit_data.at[t, 'AwayTeam']):
        pos = league.index(IDX[team])
        rows_tr.append((team, fit_data.at[t, 'Date'], fit_data.at[t, 'season'], f_[pos], sd_[pos]))
traj = pd.DataFrame(rows_tr, columns=['team', 'date', 'season', 'strength', 'sd'])
for (team, s), g in traj.groupby(['team', 'season']):
    col = HIGHLIGHT.get(team)
    if col:
        ax[1].fill_between(g['date'], g['strength'] - 1.28 * g['sd'], g['strength'] + 1.28 * g['sd'],
                           color=col, alpha=0.15)
        ax[1].plot(g['date'], g['strength'], color=col, lw=2, label=team if s == 2021 else None)
    else:
        ax[1].plot(g['date'], g['strength'], color=GREY, lw=0.6, alpha=0.5)
ax[1].axhline(0, color='black', lw=0.8)
ax[1].set(xlabel='date', ylabel='strength (shots on target vs average team)')
ax[1].legend(frameon=False, fontsize=8)
ax[1].set_title('(b) Filtered strength, 80% bands')
fig.autofmt_xdate()
fig.tight_layout()
save_fig(fig, 'fig4_kalman')
plt.show()

# CELL 6 - Out-of-sample validation and test
Parameters are re-estimated every summer on the previous season only (a rule fixed in advance),
and every forecast uses past matches only. We ask three questions: are the uncertainty bands
honest, does the filter beat simple benchmarks, and does it add anything to the market odds?

In [ ]:
import statsmodels.api as sm

WINDOW = 1                                   # seasons used in each summer re-estimation


def refit_summer(last_season, window=WINDOW):
    """Filter runs from 2021/22, but only the last `window` seasons enter the likelihood."""
    d = SHOTS_DATA[SHOTS_DATA['season'] <= last_season].reset_index(drop=True)
    mask = (d['season'] > last_season - window).to_numpy()
    return fit_params(d, mask=mask)


def rolling_elo(d, K, h):
    """Benchmarks computed with past information only: season-to-date average and a fixed-gain Elo."""
    n = len(d)
    home, away = d['HomeTeam'].to_numpy(), d['AwayTeam'].to_numpy()
    y = (d['HST'] - d['AST']).to_numpy(float)
    season = d['season'].to_numpy()
    p_mean, p_elo, elo = np.empty(n), np.empty(n), {}
    for t in range(n):
        if t == 0 or season[t] != season[t - 1]:
            tot, cnt = {}, {}
            if t > 0:
                before = set(d.loc[season == season[t - 1], 'HomeTeam'])
                now = set(d.loc[season == season[t], 'HomeTeam'])
                mean_rel = np.mean([elo[s] for s in before - now])
                for s in now - before:
                    elo[s] = mean_rel
        i, j = home[t], away[t]
        p_mean[t] = tot.get(i, 0.0) / max(cnt.get(i, 0), 1) - tot.get(j, 0.0) / max(cnt.get(j, 0), 1) + h
        p_elo[t] = elo.get(i, 0.0) - elo.get(j, 0.0) + h
        e = y[t] - p_elo[t]
        elo[i], elo[j] = elo.get(i, 0.0) + K * e, elo.get(j, 0.0) - K * e
        for team, val in ((i, y[t]), (j, -y[t])):
            tot[team] = tot.get(team, 0.0) + val
            cnt[team] = cnt.get(team, 0) + 1
    return p_mean, p_elo


def tune_elo(last_season, h, window=WINDOW):
    d = SHOTS_DATA[SHOTS_DATA['season'] <= last_season].reset_index(drop=True)
    mask = (d['season'] > last_season - window).to_numpy()
    y = (d['HST'] - d['AST']).to_numpy(float)
    return minimize_scalar(lambda K: np.mean((y - rolling_elo(d, K, h)[1])[mask] ** 2),
                           bounds=(0.001, 0.5), method='bounded').x


def fallback(frame, first, second):
    """Use Pinnacle when available, otherwise Betfair Exchange (Pinnacle is missing from Jan 2026)."""
    a = frame[first].to_numpy(float).copy()
    b = frame[second].to_numpy(float)
    miss = np.isnan(a).any(axis=1)
    a[miss] = b[miss]
    return a


def market_diff(frame):
    """Market goal-intensity difference lam_home - lam_away, with the Betfair fallback."""
    q = fallback(frame, ['PSCH', 'PSCD', 'PSCA'], ['BFECH', 'BFECD', 'BFECA'])
    q_ou = fallback(frame, ['PC>2.5', 'PC<2.5'], ['BFEC>2.5', 'BFEC<2.5'])
    out = np.full(len(frame), np.nan)
    good = ~np.isnan(q).any(axis=1) & ~np.isnan(q_ou).any(axis=1)
    if good.any():
        p = devig(q[good])
        po = devig(q_ou[good])[:, 0]
        lh, la, _ = implied_intensities(p[:, 0], p[:, 2], po)
        out[good] = lh - la
    return out


LEVELS = [0.5, 0.8, 0.9, 0.95]
results, rows5 = {}, []
for season in [2023, 2024, 2025]:
    p_ = refit_summer(season - 1)
    K_ = tune_elo(season - 1, p_['h'])
    d = SHOTS_DATA[SHOTS_DATA['season'] <= season].reset_index(drop=True)
    kk = kalman_league(d, p_)
    sel = (d['season'] == season).to_numpy()
    p_mean, p_elo = rolling_elo(d, K_, p_['h'])
    y = kk['y'][sel]
    err = {'Kalman': (y - kk['pred'][sel]) ** 2, 'home advantage only': (y - p_['h']) ** 2,
           'season-to-date mean': (y - p_mean[sel]) ** 2, 'Elo on shots': (y - p_elo[sel]) ** 2}
    zz = kk['v'][sel] / np.sqrt(kk['F'][sel])
    week = d.loc[sel, 'Date'].dt.to_period('W').to_numpy()
    row = {'season': f'{season}/{str(season + 1)[2:]}', 'R': p_['R'], 'h': p_['h'],
           'surprise variance': zz.var(), '80% coverage': np.mean(np.abs(zz) <= norm.ppf(0.9))}
    for name, e in err.items():
        row[f'RMSE {name}'] = np.sqrt(e.mean())
        if name != 'Kalman':
            dd = pd.Series(e - err['Kalman']).groupby(week).mean()
            tt = dd.mean() / dd.std(ddof=1) * np.sqrt(len(dd))
            row[f'p {name}'] = 2 * student_t.sf(abs(tt), len(dd) - 1)
    rows5.append(row)
    results[season] = dict(z=zz, err=err, week=week, date=d.loc[sel, 'Date'].to_numpy(),
                           reg=pd.DataFrame({'season': season,
                                             'goal_diff': (d['FTHG'] - d['FTAG']).to_numpy(float)[sel],
                                             'market': market_diff(d[sel]),
                                             'kalman': kk['pred'][sel]}))

tab5 = pd.DataFrame(rows5).set_index('season')
print('\nTABLE 5 - Out-of-sample forecasting (2023/24 = validation, 2024/25 and 2025/26 = test)')
print(tab5.round(3).to_string())

# does the filter add anything to the market? one constant per season, HC1 standard errors
reg = pd.concat([results[s]['reg'] for s in [2024, 2025]], ignore_index=True).dropna()
X = pd.get_dummies(reg['season'], prefix='const', dtype=float).join(reg[['market', 'kalman']])
ols = sm.OLS(reg['goal_diff'], X).fit(cov_type='HC1')
print(f'\nEncompassing regression on the test seasons ({len(reg)} matches):')
print(ols.summary2().tables[1][['Coef.', 'Std.Err.', 'z', 'P>|z|']].round(3).to_string())
print(f'correlation between the two forecasts: {reg["market"].corr(reg["kalman"]):.2f}')
print(f"-> does the Kalman filter add information to the market? "
      f"{'YES' if (ols.params['kalman'] > 0 and ols.pvalues['kalman'] < 0.05) else 'NO'}")

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
w = 0.26
for i, (season, col) in enumerate(zip([2024, 2025], [BLUE, ORANGE])):
    cov = [np.mean(np.abs(results[season]['z']) <= norm.ppf(0.5 + l / 2)) for l in LEVELS]
    ax[0].bar(np.arange(len(LEVELS)) + (i - 0.5) * w, cov, w, color=col,
              label=f'{season}/{str(season + 1)[2:]}')
ax[0].plot(np.arange(len(LEVELS)), LEVELS, 'k_', ms=22, label='nominal')
ax[0].set(xticks=np.arange(len(LEVELS)), xticklabels=[f'{int(l * 100)}%' for l in LEVELS],
          xlabel='nominal band', ylabel='observed coverage', ylim=(0, 1.05))
ax[0].legend(frameon=False, fontsize=8)
ax[0].set_title('(a) Are the uncertainty bands honest?')

base = 0.0
for name, col in [('home advantage only', GREY), ('season-to-date mean', ORANGE), ('Elo on shots', RED)]:
    for season in [2024, 2025]:
        r = results[season]
        cum = base + np.cumsum(r['err'][name] - r['err']['Kalman'])
        ax[1].plot(r['date'], cum, color=col, lw=1.8,
                   label=name if season == 2024 else None)
        base = cum[-1]
    base = 0.0
ax[1].axhline(0, color='black', lw=0.8)
ax[1].set(xlabel='date', ylabel='cumulative squared error vs Kalman')
ax[1].legend(frameon=False, fontsize=8)
ax[1].set_title('(b) Rising line = Kalman forecasts better')
fig.autofmt_xdate()
fig.tight_layout()
save_fig(fig, 'fig5_outofsample')
plt.show()

# CELL 6b - Model comparison and diagnostics

Three questions a referee will ask. Are team strengths needed at all, and does strength
drift? (LR test, AIC, BIC, and the likelihood accumulated over the sample.) How much of the
variance does the filter absorb, and how much is irreducible match noise? Are the innovations
what the model claims, that is standard normal white noise with no volatility clustering?

The profile likelihoods show how sharply each variance parameter is identified: flat near
zero for the drift Q, with a clear peak for the prior P0.

In [ ]:
# A state-space model is judged by its innovations v (the surprise of each match) and F (the
# variance the filter had announced for it). If the model is right, z = v / sqrt(F) is standard
# normal white noise. These are the usual checks, plus the model comparison by LR, AIC and BIC.
from scipy.stats import jarque_bera, probplot, skew, kurtosis
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

# --- 1) nested models: are team strengths needed, and does strength drift?
y_fit = (fit_data['HST'] - fit_data['AST']).to_numpy(float)
ll_flat = -0.5 * np.sum(np.log(2 * np.pi * y_fit.var()) + (y_fit - y_fit.mean()) ** 2 / y_fit.var())
models = {'no team strength (h + noise)': (ll_flat, 2),
          'constant strength (Q = 0)': (par_Q0['logL'], par_Q0['k']),
          'full model (Q free)': (par['logL'], par['k'])}
n_obs = len(fit_data)
tab = pd.DataFrame([{'model': m, 'logL': ll, 'parameters': k,
                     'AIC': 2 * k - 2 * ll, 'BIC': k * np.log(n_obs) - 2 * ll}
                    for m, (ll, k) in models.items()]).set_index('model')
print(f'TABLE 4b - Model comparison on {n_obs} matches (lower AIC and BIC are better)')
print(tab.round(2).to_string())
print(f'LR test for Q = 0: {LR:.2f}, p = {p_LR:.3f} (boundary case: half a chi2 with 1 d.o.f.)')
print(f'LR test for "no team strength": {2 * (par_Q0["logL"] - ll_flat):.1f} on 1 d.o.f., '
      f'p = {chi2.sf(2 * (par_Q0["logL"] - ll_flat), 1):.1e}')

# --- 2) how much of the variance does the filter absorb?
print(f'\nVariance of the shot difference: total {y_fit.var():.2f} = '
      f'explained by strengths and home advantage {kf["pred"].var():.2f} '
      f'({kf["pred"].var() / y_fit.var():.0%}) + irreducible match noise R {par["R"]:.2f}')
first, last = kf['F'][:38].mean(), kf['F'][-38:].mean()
print(f'Announced forecast variance F = (uncertainty about strengths) + R:  '
      f'first 38 matches {first:.2f} = {first - par["R"]:.2f} + {par["R"]:.2f}; '
      f'last 38 matches {last:.2f} = {last - par["R"]:.2f} + {par["R"]:.2f}')

# --- 3) are the innovations standard normal white noise?
z_in = kf['v'] / np.sqrt(kf['F'])                                  # in sample
z_out = np.concatenate([results[s]['z'] for s in [2024, 2025]])    # out of sample (test seasons)
rows = []
for label, z_ in [('in sample', z_in), ('out of sample', z_out)]:
    rows.append({'sample': label, 'n': len(z_), 'mean': z_.mean(), 'variance': z_.var(),
                 'skewness': skew(z_), 'excess kurtosis': kurtosis(z_),
                 'Jarque-Bera p': jarque_bera(z_).pvalue,
                 'Ljung-Box p (5 lags)': float(acorr_ljungbox(z_, lags=[5], return_df=True)['lb_pvalue'].iloc[0]),
                 'ARCH-LM p (5 lags)': het_arch(z_, nlags=5)[1]})
print('\nTABLE 4c - Innovations: the model assumes standard normal white noise')
print(pd.DataFrame(rows).set_index('sample').round(3).to_string())
print('mean 0 and variance 1 (bands honest), Jarque-Bera p > 0.05 (normal), '
      'Ljung-Box p > 0.05 (independent), ARCH-LM p > 0.05 (no volatility clustering, R can stay constant)')

fig, ax = plt.subplots(2, 2, figsize=(9, 6.5))

# (a) the filter at work on one team: noisy matches, filtered strength, 80% band
TEAM = 'Inter'
g = traj[(traj['team'] == TEAM) & (traj['season'] == 2021)]
obs = fit_data[((fit_data['HomeTeam'] == TEAM) | (fit_data['AwayTeam'] == TEAM))
               & (fit_data['season'] == 2021)]
y_obs = np.where(obs['HomeTeam'] == TEAM, obs['HST'] - obs['AST'] - par['h'],
                 obs['AST'] - obs['HST'] + par['h'])
ax[0, 0].axhline(0, color='black', lw=0.8)
ax[0, 0].plot(np.arange(1, len(y_obs) + 1), y_obs, 'o', ms=3.5, color=GREY, alpha=0.7,
              label='single match (noisy)')
n_g = np.arange(1, len(g) + 1)
ax[0, 0].fill_between(n_g, g['strength'] - 1.28 * g['sd'], g['strength'] + 1.28 * g['sd'],
                      color=BLUE, alpha=0.2, label='80% band')
ax[0, 0].plot(n_g, g['strength'], color=BLUE, lw=2, label='filtered strength')
ax[0, 0].set(xlabel='match of the season', ylabel='shots on target vs average',
             title=f'(a) The filter at work: {TEAM}, 2021/22')
ax[0, 0].legend(frameon=False, fontsize=7)

# (b) the Kalman gain: high when the filter knows little, settling to a steady state
gain = np.array([(kf['PP'][t][IDX[fit_data.at[t, 'HomeTeam']], IDX[fit_data.at[t, 'HomeTeam']]]
                  - kf['PP'][t][IDX[fit_data.at[t, 'HomeTeam']], IDX[fit_data.at[t, 'AwayTeam']]])
                 / kf['F'][t] for t in range(len(fit_data))])
ax[0, 1].plot(np.arange(1, len(gain) + 1), gain, color=BLUE, lw=1)
ax[0, 1].axvline(380, color=GREY, ls='--', lw=1)
ax[0, 1].text(390, gain.max() * 0.88, 'summer break', fontsize=7, color=GREY)
ax[0, 1].annotate('spikes: promoted teams,\nthe filter knows nothing yet', xy=(430, gain.max() * 0.55),
                  xytext=(500, gain.max() * 0.8), fontsize=7, color=GREY,
                  arrowprops=dict(arrowstyle='->', color=GREY, lw=0.8))
ax[0, 1].set(xlabel='match of the sample', ylabel='Kalman gain on the home team',
             title='(b) The gain falls as the filter learns')

# (c) are the innovations normal?
probplot(z_out, dist='norm', plot=ax[1, 0])
ax[1, 0].get_lines()[0].set(marker='o', ms=3, color=BLUE, alpha=0.6)
ax[1, 0].get_lines()[1].set(color=RED, lw=1.5)
ax[1, 0].set(title=f'(c) Normal? Jarque-Bera p = {jarque_bera(z_out).pvalue:.2f}',
             xlabel='theoretical quantiles', ylabel='observed quantiles')

# (d) are they independent?
lags = np.arange(1, 11)
acf = [np.corrcoef(z_out[:-k], z_out[k:])[0, 1] for k in lags]
band = 1.96 / np.sqrt(len(z_out))
ax[1, 1].bar(lags, acf, color=BLUE, width=0.6)
ax[1, 1].fill_between([0.4, 10.6], -band, band, color=GREY, alpha=0.25, label='chance (95%)')
ax[1, 1].axhline(0, color='black', lw=0.8)
ax[1, 1].set(xlabel='lag (matches)', ylabel='autocorrelation', xticks=lags,
             title='(d) Independent? Bars inside the band')
ax[1, 1].legend(frameon=False, fontsize=7)
fig.tight_layout()
save_fig(fig, 'fig4b_diagnostics')
plt.show()
# --- 4) where does the evidence come from? the likelihood accumulated over the sample
def loglik_path(par_, d):
    """Contribution of each match to the log-likelihood."""
    k = kalman_league(d, par_)
    return -0.5 * (np.log(2 * np.pi * k['F']) + k['v'] ** 2 / k['F'])


path_full = loglik_path(par, fit_data)
path_naive = -0.5 * (np.log(2 * np.pi * y_fit.var()) + (y_fit - y_fit.mean()) ** 2 / y_fit.var())
roll = pd.Series(path_full).rolling(38, min_periods=10).mean()

fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
m_axis = np.arange(1, len(fit_data) + 1)

ax[0].plot(m_axis, np.cumsum(path_full - path_naive), color=BLUE, lw=2)
ax[0].axvline(380, color=GREY, ls='--', lw=1)
ax[0].text(390, 10, 'summer break', fontsize=7, color=GREY)
ax[0].axhline(0, color='black', lw=0.8)
ax[0].set(xlabel='match of the sample', ylabel='cumulative log-likelihood gain',
          title='(a) Evidence for team strengths accumulates')

ax[1].plot(m_axis, roll, color=BLUE, lw=2, label='filter (38-match average)')
ax[1].axhline(path_naive.mean(), color=RED, ls='--', lw=1.5, label='no team strength')
ax[1].axvline(380, color=GREY, ls='--', lw=1)
ax[1].set(xlabel='match of the sample', ylabel='log-likelihood per match',
          title='(b) Forecasts improve as the filter learns')
ax[1].legend(frameon=False, fontsize=7)

x_pos = np.arange(len(tab))
ax[2].bar(x_pos - 0.2, tab['AIC'], 0.38, color=BLUE, label='AIC')
ax[2].bar(x_pos + 0.2, tab['BIC'], 0.38, color=ORANGE, label='BIC')
ax[2].set(xticks=x_pos, xticklabels=['no team\nstrength', 'constant\nstrength', 'full\nmodel'],
          ylabel='information criterion (lower is better)',
          ylim=(tab[['AIC', 'BIC']].min().min() - 40, tab[['AIC', 'BIC']].max().max() + 40))
ax[2].legend(frameon=False, fontsize=8)
ax[2].set_title('(c) Strengths are needed, drift is not')
fig.tight_layout()
save_fig(fig, 'fig4c_evidence')
plt.show()

# --- 5) how sharply are the two variance parameters identified?
grid_P0 = np.geomspace(0.3, 30, 25)
prof_P0 = np.array([log_likelihood({**par, 'P0': p}, fit_data) for p in grid_P0]) - par['logL']
P0_lo, P0_hi = grid_P0[prof_P0 >= -1.92].min(), grid_P0[prof_P0 >= -1.92].max()
print(f'\nprofile likelihood: Q in [0, {Q_max:.3f}], P0 in [{P0_lo:.2f}, {P0_hi:.2f}] at 95%')

fig, ax = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for a, grid, prof, name, best in [(ax[0], grid_Q, profile, 'Q (drift per match)', None),
                                  (ax[1], grid_P0, prof_P0, 'P0 (prior on strength)', par['P0'])]:
    a.plot(grid, prof, color=BLUE, lw=2)
    a.axhline(-1.92, color=GREY, ls=':', lw=1.2)
    if best:
        a.axvline(best, color=ORANGE, ls='--', lw=1.2, label=f'estimate {best:.2f}')
        a.legend(frameon=False, fontsize=7)
    a.set(xscale='log', xlabel=name, ylim=(-12, 0.5))
ax[0].set_ylabel('log-likelihood (0 = maximum)')
ax[0].set_title('(a) Flat near zero: no drift')
ax[1].set_title('(b) A clear peak: the prior is identified')
fig.tight_layout()
save_fig(fig, 'fig4d_profiles')
plt.show()

# CELL 7 - Betting backtest with Friday odds

Everything a bettor can know on Friday: Pinnacle Friday odds as the fair price, the
Kalman forecast computed on Friday, and Bet365 Friday odds to bet on. Closing odds are
used ONLY to measure CLV, the price we got compared with the final market price.

The filter is allowed to shift the market probabilities, not to rebuild them: one weight
per outcome, estimated on earlier seasons only.

Nothing here depends on the size of the bankroll. The edge is measured by the **yield**
(profit per euro staked) and by the **CLV**; how fast that edge can grow a bankroll is
the Kelly **growth rate**, which is a property of the strategy. A fixed starting capital
would only rescale the picture, and would make an arbitrary stake rule look like a result.

In [ ]:
KELLY_CAP = 0.02                         # never more than 2% of the bankroll on one bet
MIN_BET = 2.0                            # typical minimum accepted by a bookmaker, in euro
TRAIN_START, TEST_SEASONS = 2022, [2024, 2025]
bt = df[df['season'] >= TRAIN_START].reset_index(drop=True)

p_friday = np.full((len(bt), 3), np.nan)
good = ~np.isnan(fallback(bt, ['PSH', 'PSD', 'PSA'], ['BFEH', 'BFED', 'BFEA'])).any(axis=1)
p_friday[good] = devig(fallback(bt, ['PSH', 'PSD', 'PSA'], ['BFEH', 'BFED', 'BFEA'])[good])
p_close = np.full((len(bt), 3), np.nan)
good_c = ~np.isnan(fallback(bt, ['PSCH', 'PSCD', 'PSCA'], ['BFECH', 'BFECD', 'BFECA'])).any(axis=1)
p_close[good_c] = devig(fallback(bt, ['PSCH', 'PSCD', 'PSCA'], ['BFECH', 'BFECD', 'BFECA'])[good_c])
odds_b365 = bt[['B365H', 'B365D', 'B365A']].to_numpy(float)
outcome = np.select([bt['FTHG'] > bt['FTAG'], bt['FTHG'] == bt['FTAG']], [0, 1], 2)


def collection_day(date):
    """Odds are collected on Friday for weekend matches, on Tuesday for midweek rounds."""
    wd = date.weekday()
    return date - pd.Timedelta(days=wd - 1 if wd in (1, 2, 3) else (wd - 4) % 7)


kalman_friday = np.full(len(bt), np.nan)
for season in [2022, 2023] + TEST_SEASONS:
    p_ = refit_summer(season - 1)
    d = SHOTS_DATA[SHOTS_DATA['season'] <= season].reset_index(drop=True)
    kk = kalman_league(d, p_)
    dates = d['Date'].to_numpy()
    key = {(a, b): i for i, (a, b) in enumerate(zip(bt['Date'], bt['HomeTeam']))}
    for t in np.where((d['season'] == season).to_numpy())[0]:
        cut = np.datetime64(collection_day(d.at[t, 'Date']))
        b_state = kk['B'][np.searchsorted(dates, cut, side='left')]     # state before that day
        i, j = IDX[d.at[t, 'HomeTeam']], IDX[d.at[t, 'AwayTeam']]
        kalman_friday[key[(d.at[t, 'Date'], d.at[t, 'HomeTeam'])]] = b_state[i] - b_state[j] + p_['h']

usable = (~np.isnan(p_friday).any(axis=1) & ~np.isnan(p_close).any(axis=1)
          & ~np.isnan(odds_b365).any(axis=1) & ~np.isnan(kalman_friday))
season_bt = bt['season'].to_numpy()
print(f'usable matches per season: {bt.loc[usable, "season"].value_counts().sort_index().to_dict()}')


def shifted_probs(p_mkt, x, w):
    """log p = log p_market + w * kalman forecast, renormalised. w = 0 gives the market back."""
    z = np.log(p_mkt) + np.outer(x, np.r_[w[0], 0.0, w[1]])
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


p_filter = np.full((len(bt), 3), np.nan)
for season in TEST_SEASONS:
    tr = np.where(usable & (season_bt < season))[0]
    te = np.where(usable & (season_bt == season))[0]
    w = minimize(lambda w: -np.sum(np.log(shifted_probs(p_friday[tr], kalman_friday[tr], w)
                                          [np.arange(len(tr)), outcome[tr]])),
                 [0.0, 0.0], method='Nelder-Mead').x
    p_filter[te] = shifted_probs(p_friday[te], kalman_friday[te], w)
    print(f'filter weight estimated on {TRAIN_START}..{season - 1}: '
          f'w(home) = {w[0]:+.4f}, w(away) = {w[1]:+.4f}')

test_rows = np.where(usable & np.isin(season_bt, TEST_SEASONS))[0]
train_rows = np.where(usable & (season_bt < TEST_SEASONS[0]))[0]


def select_bets(p, rows, rule='value'):
    """One bet per match: the outcome with the highest expected value, if positive."""
    ev = p[rows] * odds_b365[rows] - 1
    k = ev.argmax(axis=1) if rule == 'value' else p[rows].argmax(axis=1)
    best = ev[np.arange(len(rows)), k]
    play = best > 0 if rule == 'value' else np.ones(len(rows), bool)
    r, k = rows[play], k[play]
    odds, won = odds_b365[r, k], outcome[r] == k
    return pd.DataFrame({'date': bt.loc[r, 'Date'].to_numpy(), 'odds': odds, 'ev': best[play],
                         'won': won, 'draw': k == 1,
                         'unit': np.where(won, odds - 1, -1.0),          # profit per euro staked
                         'clv': odds * p_close[r, k] - 1}).assign(
        week=lambda x: pd.to_datetime(x['date']).dt.to_period('W'))


# how much of the estimated edge survives until kick-off? estimated on TRAINING seasons only
train_bets = select_bets(p_friday, train_rows)
SHRINK = float(np.clip(train_bets['clv'].mean() / train_bets['ev'].mean(), 0, 1))
print(f'edge shrinkage from {TRAIN_START}..{TEST_SEASONS[0] - 1}: CLV / EV = {SHRINK:.2f} '
      f'(the market moves against part of the estimated edge before kick-off)')

strategies = {'market only': select_bets(p_friday, test_rows),
              'market + filter': select_bets(p_filter, test_rows),
              'favourite (control)': select_bets(p_friday, test_rows, rule='favourite')}
weeks = pd.PeriodIndex(sorted(bt.loc[test_rows, 'Date'].dt.to_period('W').unique()), freq='W')
SEASONS_TESTED = len(TEST_SEASONS)

# --- TABLE 6: the edge. These measures do not depend on how much we stake.
rows6 = []
for name, b in strategies.items():
    t_y = b['unit'].mean() / b['unit'].std(ddof=1) * np.sqrt(len(b))
    t_c = b['clv'].mean() / b['clv'].std(ddof=1) * np.sqrt(len(b))
    rows6.append({'strategy': name, 'bets': len(b), 'bets per season': len(b) / SEASONS_TESTED,
                  'share on the draw': b['draw'].mean(), 'mean odds': b['odds'].mean(),
                  'mean EV': b['ev'].mean(),
                  'yield': b['unit'].mean(), 't (yield)': t_y, 'p (yield)': student_t.sf(t_y, len(b) - 1),
                  'CLV': b['clv'].mean(), 't (CLV)': t_c, 'p (CLV)': student_t.sf(t_c, len(b) - 1)})
tab6 = pd.DataFrame(rows6).set_index('strategy')
print(f'\nTABLE 6 - Is there an edge? {len(weeks)} matchdays, {len(test_rows)} matches. '
      f'One-sided p-values; with three strategies the corrected threshold is {0.05 / 3:.3f}')
print(tab6.round(3).to_string())

# --- paired test on return per euro staked: identical under any staking rule
pair = pd.DataFrame({n: b.groupby('week')['unit'].mean() for n, b in strategies.items()}).reindex(weeks)
gap = (pair['market + filter'] - pair['market only']).dropna()
t_gap = gap.mean() / gap.std(ddof=1) * np.sqrt(len(gap))
print(f'\npaired test on the {len(gap)} matchdays where both strategies bet: '
      f'{gap.mean():+.3f} per euro staked, t = {t_gap:+.2f}, '
      f'one-sided p = {student_t.sf(t_gap, len(gap) - 1):.3f}')


def kelly_growth(b, fraction, shrink):
    """Expected log growth per bet under fractional Kelly, and the stakes it implies.
    This is a property of the strategy: it does not depend on the size of the bankroll."""
    o, edge = b['odds'].to_numpy(), np.clip(b['ev'].to_numpy(), 0, None) * shrink
    f = np.minimum(fraction * edge / (o - 1), KELLY_CAP)
    p_true = (1 + edge) / o                                     # win probability implied by that edge
    g = p_true * np.log1p(f * (o - 1)) + (1 - p_true) * np.log1p(-f)
    return g.mean(), f


# --- TABLE 6b: how fast can this edge grow a bankroll, whatever its size?
rows6b = []
for shrink, label in [(1.0, 'EV as estimated'), (SHRINK, f'EV shrunk ({SHRINK:.2f})')]:
    for fraction in [1.0, 0.5, 0.25]:
        g, f = kelly_growth(strategies['market only'], fraction, shrink)
        rows6b.append({'edge used': label, 'Kelly fraction': fraction,
                       'growth per bet': g, 'growth per season':
                           np.expm1(g * len(strategies['market only']) / SEASONS_TESTED),
                       'median stake (% of bankroll)': 100 * np.median(f)})
tab6b = pd.DataFrame(rows6b).set_index(['edge used', 'Kelly fraction'])
print('\nTABLE 6b - Growth of the "market only" strategy (the one with positive CLV), '
      'independent of the bankroll')
print(tab6b.round(5).to_string())

# --- TABLE 6c: can it be implemented at all with a small bankroll?
_, f_q = kelly_growth(strategies['market only'], 0.25, SHRINK)
rows6c = [{'bankroll (euro)': bank, 'median stake (euro)': bank * np.median(f_q),
           'largest stake (euro)': bank * f_q.max(),
           f'bets below the {MIN_BET:.0f} euro minimum': (bank * f_q < MIN_BET).mean()}
          for bank in [100, 500, 1000, 5000]]
print(f'\nTABLE 6c - Quarter Kelly on the shrunk edge: what it means in euro '
      f'(bankroll needed for a {MIN_BET:.0f} euro median stake: '
      f'{MIN_BET / np.median(f_q):,.0f} euro)')
print(pd.DataFrame(rows6c).set_index('bankroll (euro)').round(3).to_string())

fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
for name, col, ls in zip(strategies, [ORANGE, BLUE, RED], ['-', '-', ':']):
    b = strategies[name]
    cum = b.groupby('week')['unit'].sum().reindex(weeks, fill_value=0.0).cumsum()
    ax[0].plot(weeks.to_timestamp(how='end'), cum.to_numpy(), color=col, lw=2, ls=ls,
               label=f'{name} ({len(b)} bets)')
ax[0].axhline(0, color='black', lw=0.8)
ax[0].set(xlabel='date', ylabel='cumulative profit (units staked)')
ax[0].legend(frameon=False, fontsize=7)
ax[0].set_title('(a) Profit in units, free of any stake rule')

fr = np.linspace(0.05, 2.0, 40)
for shrink, col, label in [(1.0, GREY, 'EV as estimated'), (SHRINK, BLUE, f'EV shrunk ({SHRINK:.2f})')]:
    growth = [np.expm1(kelly_growth(strategies['market only'], x, shrink)[0]
                       * len(strategies['market only']) / SEASONS_TESTED) for x in fr]
    ax[1].plot(fr, 100 * np.array(growth), color=col, lw=2, label=label)
ax[1].axhline(0, color='black', lw=0.8)
ax[1].axvline(1.0, color=RED, ls='--', lw=1, label='full Kelly')
ax[1].set(xlabel='Kelly fraction', ylabel='growth per season (%)')
ax[1].legend(frameon=False, fontsize=7)
ax[1].set_title('(b) Even full Kelly grows the bankroll by under 1%')

names = list(strategies)
ax[2].bar(np.arange(3) - 0.2, [tab6.loc[n, 'yield'] for n in names], 0.38, color=BLUE, label='yield')
ax[2].bar(np.arange(3) + 0.2, [tab6.loc[n, 'CLV'] for n in names], 0.38, color=ORANGE, label='CLV')
ax[2].axhline(0, color='black', lw=0.8)
ax[2].set(xticks=np.arange(3), xticklabels=['market', 'market\n+ filter', 'favourite'],
          ylabel='per euro staked')
ax[2].legend(frameon=False, fontsize=8)
ax[2].set_title('(c) A positive yield with negative CLV is luck')
fig.autofmt_xdate()
fig.tight_layout()
save_fig(fig, 'fig6_betting')
plt.show()

# CELL 8 - Application: reading the current season
The filter does not beat the market, but it measures team strength with a stated
error. That is enough to separate quality from luck: how many points a team should
have given the market, and how many goals it scored given its shots. Conversion
luck does not persist, so large gaps tend to disappear.

In [ ]:
CURRENT = 2026
cur = df[df['season'] == CURRENT].reset_index(drop=True)
prev_mean = df.loc[df['season'] == CURRENT - 1, ['HST', 'AST']].stack().mean()
SCALE = prev_mean / cur[['HST', 'AST']].stack().mean()     # shots are counted differently this season
cur['HST_raw'], cur['AST_raw'] = cur['HST'], cur['AST']
cur['HST'], cur['AST'] = cur['HST'] * SCALE, cur['AST'] * SCALE
print(f'{CURRENT}/{str(CURRENT + 1)[2:]}: {len(cur)} matches played, last on {cur["Date"].max().date()}, '
      f'shots rescaled by {SCALE:.3f}')

par_cur = refit_summer(CURRENT - 1)
TEAMS = sorted(set(df.loc[df['season'] >= 2021, 'HomeTeam']) | set(df.loc[df['season'] >= 2021, 'AwayTeam']))
IDX = {t: i for i, t in enumerate(TEAMS)}
IN_SEASON = {s: set(g['HomeTeam'].map(IDX)) for s, g in df[df['season'] >= 2021].groupby('season')}
hist = pd.concat([SHOTS_DATA, cur], ignore_index=True).sort_values('Date', kind='stable').reset_index(drop=True)
kc = kalman_league(hist, par_cur)

# state AFTER the last match (the filter stores the state BEFORE each match)
last = len(hist) - 1
i, j = IDX[hist.at[last, 'HomeTeam']], IDX[hist.at[last, 'AwayTeam']]
b_now, P_now = kc['B'][last].copy(), kc['PP'][last].copy()
gain = (P_now[:, i] - P_now[:, j]) / kc['F'][last]
b_now, P_now = b_now + gain * kc['v'][last], P_now - np.outer(gain, gain) * kc['F'][last]
league = sorted(IN_SEASON[CURRENT])
C = -np.ones((len(league), len(TEAMS))) / len(league)
C[np.arange(len(league)), league] += 1
strength = pd.DataFrame({'strength': C @ b_now, 'sd': np.sqrt(np.diag(C @ P_now @ C.T))},
                        index=[TEAMS[i] for i in league])

p_cur = devig(fallback(cur, ['PSCH', 'PSCD', 'PSCA'], ['BFECH', 'BFECD', 'BFECA']))
sides = []
for home in (True, False):
    pw = p_cur[:, 0] if home else p_cur[:, 2]
    sides.append(pd.DataFrame({
        'team': cur['HomeTeam'] if home else cur['AwayTeam'],
        'points': np.select([cur['FTHG'] > cur['FTAG'] if home else cur['FTAG'] > cur['FTHG'],
                             cur['FTHG'] == cur['FTAG']], [3, 1], 0),
        'exp_points': 3 * pw + p_cur[:, 1], 'exp_points2': 9 * pw + p_cur[:, 1],
        'GF': cur['FTHG'] if home else cur['FTAG'], 'GA': cur['FTAG'] if home else cur['FTHG'],
        'SF': cur['HST_raw'] if home else cur['AST_raw'],
        'SA': cur['AST_raw'] if home else cur['HST_raw']}))
side = pd.concat(sides)
tab = side.groupby('team').sum(numeric_only=True)
var_points = side.assign(v=side['exp_points2'] - side['exp_points'] ** 2).groupby('team')['v'].sum()
tab['z_points'] = (tab['points'] - tab['exp_points']) / np.sqrt(var_points)
conv_rate = (cur['FTHG'].sum() + cur['FTAG'].sum()) / (cur['HST_raw'].sum() + cur['AST_raw'].sum())
luck = (tab['GF'] - conv_rate * tab['SF']) - (tab['GA'] - conv_rate * tab['SA'])
tab['z_conversion'] = luck / np.sqrt(conv_rate * (1 - conv_rate) * (tab['SF'] + tab['SA']))
tab = tab.join(strength)
tab['rank_points'] = tab['points'].rank(ascending=False, method='min').astype(int)
tab['rank_filter'] = tab['strength'].rank(ascending=False, method='min').astype(int)

Z_BONF = norm.ppf(1 - 0.05 / (2 * len(tab)))
tab7 = tab.sort_values('strength', ascending=False)[
    ['points', 'exp_points', 'z_points', 'rank_points', 'rank_filter', 'strength', 'sd', 'z_conversion']]
print(f'\nTABLE 7 - Reading the {CURRENT}/{str(CURRENT + 1)[2:]} season after {len(cur)} matches')
print(f'|z| > 1.96 is unusual for one team, but with {len(tab)} teams the corrected threshold is '
      f'{Z_BONF:.2f}')
print(tab7.round(2).to_string())

fig, ax = plt.subplots(1, 2, figsize=(9.5, 4.2))
sc = ax[0].scatter(tab['strength'], tab['points'] / (len(cur) * 2 / len(tab)),
                   c=tab['z_conversion'], cmap='coolwarm', vmin=-2.5, vmax=2.5,
                   s=45, edgecolor='black', linewidth=0.5, zorder=3)
ax[0].errorbar(tab['strength'], tab['points'] / (len(cur) * 2 / len(tab)),
               xerr=1.28 * tab['sd'], fmt='none', ecolor=GREY, elinewidth=0.8, zorder=2)
for name, r in tab.iterrows():
    ax[0].annotate(name, (r['strength'], r['points'] / (len(cur) * 2 / len(tab))),
                   fontsize=6.5, xytext=(0, 7), textcoords='offset points', ha='center')
fig.colorbar(sc, ax=ax[0], label='conversion luck (z)')
ax[0].set(xlabel='filtered strength (shots on target vs average)', ylabel='points per match')
ax[0].set_title('(a) Who is good, who is lucky')

ax[1].axhline(0, color='black', lw=0.8)
ax[1].axvline(0, color='black', lw=0.8)
for lim, ls in [(1.96, ':'), (Z_BONF, '--')]:
    ax[1].add_patch(plt.Rectangle((-lim, -lim), 2 * lim, 2 * lim, fill=False, ls=ls,
                                  edgecolor=GREY, lw=1))
ax[1].scatter(tab['strength'] / tab['sd'], tab['z_conversion'], s=40, color=BLUE,
              edgecolor='black', linewidth=0.5)
for name, r in tab.iterrows():
    ax[1].annotate(name, (r['strength'] / r['sd'], r['z_conversion']), fontsize=6.5,
                   xytext=(0, 7), textcoords='offset points', ha='center')
ax[1].set(xlabel='strength / its standard error', ylabel='conversion luck (z)')
ax[1].set_title('(b) Inside the dashed box, chance explains it')
fig.tight_layout()
save_fig(fig, 'fig7_application')
plt.show()

# CELL 9 - Summary of the study

In [ ]:
summary_rows = [
    ('Market margin (Pinnacle closing)', f'{margin.mean():.3f}', 'small and stable'),
    ('Market calibration', f'slope {slope:.2f}, R2 {r2:.2f}', 'well calibrated'),
    ('Draw mispriced by independent Poisson',
     f'{100 * (m["pX_poisson"] - m["pX"]).mean():.2f} pp', 'goals are not independent'),
    ('Implied Dixon-Coles rho', f'{RHO:.3f}', 'stable across seasons'),
    ('Persistence of gaps vs market', f'min p = {tab3["p-value"].min():.3f}',
     'no momentum the market misses'),
    ('Strength drift Q', f'{par["Q"]:.4f} (95% up to {Q_max:.3f})', 'strength is nearly constant'),
    ('Kalman vs naive benchmarks', 'p < 0.05', 'filter forecasts better'),
    ('Kalman vs tuned Elo', f'p = {tab5.loc["2024/25", "p Elo on shots"]:.2f}', 'equivalent'),
    ('Kalman added to closing odds', f'coef {ols.params["kalman"]:+.3f}, p = {ols.pvalues["kalman"]:.2f}',
     'adds nothing'),
    ('Filter weight on Friday odds', f'w = {w[0]:+.3f}, {w[1]:+.3f}', 'negative: no real signal'),
    ('Best CLV among strategies', f'{tab6["CLV"].max():+.3f}', 'only price-vs-price is positive'),
]
print('\nTABLE 8 - Summary')
print(pd.DataFrame(summary_rows, columns=['result', 'value', 'reading']).to_string(index=False))
print(f'\nFigures saved in ./{FIG_DIR} as PNG and PDF.')

# CELL 10 - The Friday card

The weekly use of the model. After refreshing the CSV, this cell reads the fixtures that
have not been played yet, prints what the filter expects and how uncertain it is, and -
if you fill in the odds - the expected value and the stake implied by fractional Kelly.

Remember what the study found: the filter does not beat the market, so a bet is justified
only by the price comparison (a bookmaker priced worse than the reference), never by the
filter alone. The filter is there to read the fixture and to flag what deserves a check.

In [ ]:
# What the filter says about the matches that have NOT been played yet, and what to do with
# the odds. Run this every Friday after refreshing the CSV; the cells above must have been run.
# Fixtures: taken from a calendar file if present, otherwise written by hand in FIXTURES.
# Odds: optional. 'fair' = Pinnacle or Betfair Exchange (the reference price),
#       'bet'  = the bookmaker you can actually bet with. Order is always (home, draw, away).
CALENDAR = os.path.join(DATA_DIR, 'calendario_serieA_2026-27.csv')
FIXTURES = [('Sassuolo', 'Lazio'), ('Roma', 'Cagliari')]          # used only if the file is missing
ODDS = {
    # ('Inter', 'Fiorentina'): dict(fair=(1.36, 5.20, 9.00), bet=(1.33, 5.25, 9.50)),
}
KELLY_FRACTION = 0.25

played = set(zip(cur['HomeTeam'], cur['AwayTeam']))
if os.path.exists(CALENDAR):
    cal = pd.read_csv(CALENDAR)
    cal['Data'] = pd.to_datetime(cal['Data'], dayfirst=True)
    todo = cal[[pair not in played for pair in zip(cal['HomeTeam'], cal['AwayTeam'])]]
    matchday = int(todo['Giornata'].min())
    fixtures = list(todo.loc[todo['Giornata'] == matchday, ['HomeTeam', 'AwayTeam']]
                    .itertuples(index=False, name=None))
    when = todo.loc[todo['Giornata'] == matchday, 'Data'].iloc[0].date()
    print(f'matchday {matchday} ({when}), {len(fixtures)} fixtures, '
          f'{len(cur)} matches already in the data')
else:
    fixtures = FIXTURES
    print(f'no calendar file: using the {len(fixtures)} fixtures written in FIXTURES')

rows = []
for home, away in fixtures:
    i, j = IDX[home], IDX[away]
    pred = b_now[i] - b_now[j] + par_cur['h']
    sd_strength = np.sqrt(P_now[i, i] + P_now[j, j] - 2 * P_now[i, j] + 2 * par_cur['Q'])
    sd_match = np.sqrt(sd_strength ** 2 + par_cur['R'])
    row = {'home': home, 'away': away, 'Kalman (shots)': pred, 'sd (strength)': sd_strength,
           '80% band': f'[{pred - 1.2816 * sd_match:+.1f}, {pred + 1.2816 * sd_match:+.1f}]',
           'home luck (z)': tab.loc[home, 'z_conversion'], 'away luck (z)': tab.loc[away, 'z_conversion']}
    o = ODDS.get((home, away))
    if o:
        fair = np.array(o['fair'], float)
        bet = np.array(o['bet'], float)
        p_fair = (1 / fair) / np.sum(1 / fair)
        ev = p_fair * bet - 1
        k = int(np.argmax(ev))
        stake = min(KELLY_FRACTION * max(ev[k], 0) * SHRINK / (bet[k] - 1), 0.02)
        row |= {'best outcome': ['1', 'X', '2'][k], 'EV': ev[k],
                'stake (% of bankroll)': 100 * stake if ev[k] > 0 else 0.0,
                'verdict': 'BET' if ev[k] > 0 else 'no value'}
    rows.append(row)
card = pd.DataFrame(rows)
print(f'\nFRIDAY CARD (filter parameters: R = {par_cur["R"]:.2f}, h = {par_cur["h"]:+.2f}; '
      f'edge shrinkage {SHRINK:.2f}, {KELLY_FRACTION:g} Kelly, cap 2%)')
print(card.round(2).to_string(index=False))
if not ODDS:
    print('\nFill ODDS above to get expected value and stakes. Without odds this is a reading of '
          'the fixtures, not a betting suggestion: the study found no edge in the filter itself.')

fig, ax = plt.subplots(figsize=(7, 0.45 * len(card) + 1.6))
y = np.arange(len(card))[::-1]
ax.errorbar(card['Kalman (shots)'], y, xerr=1.2816 * card['sd (strength)'], fmt='o',
            color=BLUE, ms=6, capsize=3, label='filtered strength, 80% band')
for yi, (_, r) in zip(y, card.iterrows()):
    sd_match = np.sqrt(r['sd (strength)'] ** 2 + par_cur['R'])
    ax.plot([r['Kalman (shots)'] - 1.2816 * sd_match, r['Kalman (shots)'] + 1.2816 * sd_match],
            [yi, yi], color=GREY, lw=1, alpha=0.5, zorder=0)
ax.plot([], [], color=GREY, lw=1, alpha=0.5, label='single match, 80% band')
ax.axvline(0, color='black', lw=0.8)
ax.set(yticks=y, yticklabels=[f'{r.home} - {r.away}' for r in card.itertuples()],
       xlabel='expected shots on target (home - away)')
ax.legend(frameon=False, fontsize=8, loc='lower right')
ax.set_title('Next matchday: what the filter expects, and how uncertain it is')
fig.tight_layout()
save_fig(fig, 'fig8_friday_card')
plt.show()